# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [6]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-5-mini'
MODEL_LLAMA = 'llama3.2:1b'

OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

system_prompt = """
You are a senior software engineer and technical educator.

Your task is to take a technical question and respond with a clear, structured, and accurate explanation.

Guidelines:
	•	Carefully analyze the question before answering.
	•	Provide a concise but thorough explanation.
	•	Break down complex concepts into simple, understandable parts.
	•	Use examples when helpful.
	•	If relevant, include code snippets to illustrate the explanation.
	•	Prefer TypeScript for code examples unless another language is explicitly requested.
	•	Avoid unnecessary jargon. When technical terms are required, explain them clearly.
	•	If the question is ambiguous, state reasonable assumptions and proceed.
	•	Focus on correctness, clarity, and practical understanding.

Your goal is not just to answer the question, but to ensure the reader truly understands the concept.
"""

API key looks good so far


In [9]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [10]:

def stream_answer(client, model):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [11]:
# Get gpt-5-mini to answer, with streaming
stream_answer(openai, MODEL_GPT)

This line is Python inside a generator function:

yield from {book.get("author") for book in books if book.get("author")}

What it does, step by step
- The set comprehension {book.get("author") for book in books if book.get("author")} builds a Python set of author values:
  - For each book in books it calls book.get("author").
  - The if book.get("author") filters out falsy values (None, empty string, 0, etc.).
  - The set removes duplicate authors.
  - The comprehension is evaluated eagerly: the whole set is constructed before yielding anything.
- yield from <iterable> then iterates over that set and yields each element to the caller of the generator.

Net effect: the generator yields each unique, truthy author found in books (in some implementation-dependent order).

Why this matters (pros and cons)
- Pros:
  - Duplicates are removed automatically because of the set.
  - The code is short.

- Cons / caveats:
  - Order is not preserved. Sets are unordered, so yield order is not guaranteed.
  - The comprehension is eager: the complete set is built in memory before any yields. For very large books, that may be undesirable.
  - book.get("author") is called twice per book (once in the value expression and once in the if), which is slightly inefficient.
  - The if filters out all falsy values; if you only want to exclude None but keep empty strings, this will wrongly drop them.

Better alternatives (depending on intent)

1) If you want unique authors but preserve the order they appear in books (and still filter out falsy values), a simple way is to use dict.fromkeys (Python 3.7+ preserves insertion order):

yield from dict.fromkeys(
    (book.get("author") for book in books if book.get("author"))
)

This builds keys in insertion order and yields unique authors in that order.

2) If you want to yield lazily (no full set built) and allow duplicates:

yield from (book.get("author") for book in books if book.get("author"))

3) If you want unique authors, preserve order, and be lazy (no double .get), do a streaming seen set:

def unique_authors(books):
    seen = set()
    for book in books:
        author = book.get("author")
        if author and author not in seen:
            seen.add(author)
            yield author

# then:
yield from unique_authors(books)

4) If you want to avoid calling .get twice inside a comprehension (Python 3.8+), you can use the walrus operator:

yield from {author for book in books if (author := book.get("author"))}

(That still builds a set and loses order.)

Summary
- The original line yields each unique, truthy author from books, but does so by building a set first (eager) and yields in arbitrary order.
- Choose an alternative depending on whether you need order preserved, memory efficiency, laziness, or to avoid duplicate .get calls.

In [ ]:
# Get Llama 3.2 to answer
stream_answer(ollama, MODEL_LLAMA)

This piece of code appears to be using Python's generator feature to iterate over a list-like object, specifically an iterable (like a list or tuple), where each iteration yields either one element at a time or the entire item as output. Here is a breakdown:

- `yield from`: This keyword allows the function to yield elements instead of directly returning them.
- `{book.get("author") ...}`: The code inside these curly braces appears to be iterating over an iterable containing objects (`books` in this context) and retrieving their "author" value with another method (`get("")`. It's assumed that `{"author" : authorValue, ...}` is a structure where "authorValue" holds the string representing the authors of books.

So, when written correctly:

-  `yield from (book.get("author") for book in books if book.get("author"))`

- The function does not return any value and instead returns an iterator that produces each item at each stage. This is because it uses a generator technique in Python, which can be more memory-efficient, especially when dealing with large datasets.

The original code:

```python
for book in books:
    if book.get("author"):
        yield from (book.get("author"))
```

- Works exactly the same way: Retrieves author of each book, yields them until there are no more authors.

However, this line:

- `yield from` is used instead of directly appending elements to an array because generators yield one element at a time. That operation can waste memory when doing so on large datasets due to Python's slow GIL or memory-intensive operations. 

Here's the code with example data that it might be applied against.

```python
def process_books():
    books = [
        {"id": 1, "title": "Book_Author", "author": "Author_A"},
        {"id": 2, "title": "Book_B_Artist", "author": "Author_b"},
        {"id": None , "title": "Book_C_Architect" }
    ]

    results = [book.get("author") for book in books]

    # We use yield from to yield each author that exists
    yielded_data = (result for result in results if result is not None)
    
    for author in yielded_data:
        print("Author whose text has been output:", author, ("...".join([f"Book_{index + 1}" for index, text in enumerate({book.get("title") for book in books} if book) for _ in range(4)])))

process_books()
```